# Multiple linear regression from scratch with the normal equation

Diabetes data: 442 patients, 10 inputs, target = disease progression one year later.

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [2]:
X, y = load_diabetes(return_X_y=True)
print(X.shape, y.shape)          # (442, 10) (442,)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

(442, 10) (442,)


## scikit-learn first

In [3]:
reg = LinearRegression().fit(X_train, y_train)
print(r2_score(y_test, reg.predict(X_test)))
print(reg.coef_.round(2), round(reg.intercept_, 2))

0.4399338661568968
[  -9.16 -205.45  516.69  340.62 -895.55  561.22  153.89  126.73  861.13
   52.42] 151.88


## Our own class

In [4]:
class MyMLR:
    def fit(self, X, y):
        # add the column of 1s for the intercept, at position 0
        Xb = np.insert(X, 0, 1, axis=1)
        # normal equation: beta = (X^T X)^-1 X^T y
        betas = np.linalg.inv(Xb.T @ Xb) @ Xb.T @ y
        self.intercept_ = betas[0]
        self.coef_ = betas[1:]
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

mine = MyMLR().fit(X_train, y_train)
print(r2_score(y_test, mine.predict(X_test)))
print(mine.coef_.round(2), round(mine.intercept_, 2))

0.439933866156897
[  -9.16 -205.45  516.69  340.62 -895.55  561.22  153.89  126.73  861.13
   52.42] 151.88


In [5]:
# largest difference from scikit-learn: tiny rounding only
np.abs(mine.coef_ - reg.coef_).max()

np.float64(3.296918293926865e-12)

## A safer way to solve: no explicit inverse

In [6]:
Xb = np.insert(X_train, 0, 1, axis=1)
b_solve = np.linalg.solve(Xb.T @ Xb, Xb.T @ y_train)      # solves X^T X beta = X^T y directly
b_lstsq = np.linalg.lstsq(Xb, y_train, rcond=None)[0]     # least squares, what libraries use
print(np.abs(b_solve[1:] - reg.coef_).max(), np.abs(b_lstsq[1:] - reg.coef_).max())

8.640199666842818e-12 5.5706550483591855e-12


## Checks used in the Note: s1 and s2, and the accuracy of inv, solve and lstsq

In [7]:
from sklearn.linear_model import Ridge
names = ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
C = np.corrcoef(X_train.T)
print('correlation of s1 and s2:', C[4, 5].round(3))                   # about 0.895
print('VIF:', dict(zip(names, np.diag(np.linalg.inv(C)).round(1))))   # s1 about 56, s2 about 37, others mostly under 10

# refit on 200 bootstrap samples: how much does each coefficient move?
rng = np.random.default_rng(0)
B = []
for _ in range(200):
    i = rng.integers(0, len(X_train), len(X_train))
    B.append(LinearRegression().fit(X_train[i], y_train[i]).coef_)
B = np.array(B)
print('std of each coefficient:', dict(zip(names, B.std(axis=0).round(0))))   # s1 about 464, s2 about 369, bmi about 77
print('correlation of the s1 and s2 coefficients:', np.corrcoef(B[:, 4], B[:, 5])[0, 1].round(2))  # about -0.97
print('ridge, alpha=0.1:', dict(zip(names, Ridge(alpha=0.1).fit(X_train, y_train).coef_.round(0))))  # s1 about -73, s2 about -81

# a design with one column almost a copy of another; the true betas are known
rng = np.random.default_rng(1)
for eps in [1e-3, 1e-5, 1e-7]:
    Z = np.c_[np.ones(200), rng.normal(size=(200, 3))]
    Z = np.c_[Z, Z[:, 1] + eps * rng.normal(size=200)]
    true = np.array([1., 2, 3, 4, 5]); t = Z @ true
    err = lambda b: np.abs(b - true).max()
    print(eps, 'inv', err(np.linalg.inv(Z.T @ Z) @ Z.T @ t), 'solve', err(np.linalg.solve(Z.T @ Z, Z.T @ t)),
          'lstsq', err(np.linalg.lstsq(Z, t, rcond=None)[0]))


correlation of s1 and s2: 0.895
VIF: {'age': np.float64(1.3), 'sex': np.float64(1.3), 'bmi': np.float64(1.6), 'bp': np.float64(1.5), 's1': np.float64(56.5), 's2': np.float64(37.4), 's3': np.float64(14.8), 's4': np.float64(9.4), 's5': np.float64(9.9), 's6': np.float64(1.5)}


std of each coefficient: {'age': np.float64(71.0), 'sex': np.float64(65.0), 'bmi': np.float64(77.0), 'bp': np.float64(73.0), 's1': np.float64(464.0), 's2': np.float64(369.0), 's3': np.float64(218.0), 's4': np.float64(181.0), 's5': np.float64(174.0), 's6': np.float64(67.0)}
correlation of the s1 and s2 coefficients: -0.97
ridge, alpha=0.1: {'age': np.float64(7.0), 'sex': np.float64(-172.0), 'bmi': np.float64(486.0), 'bp': np.float64(315.0), 's1': np.float64(-73.0), 's2': np.float64(-81.0), 's3': np.float64(-174.0), 's4': np.float64(84.0), 's5': np.float64(484.0), 's6': np.float64(74.0)}
0.001 inv 1.107935077016009e-09 solve 5.4569682106375694e-12 lstsq 4.3520742565306136e-14
1e-05 inv 2.9672948585357517e-05 solve 8.967589227637518e-05 lstsq 1.0542677841840487e-11
1e-07 inv 0.07000169693492353 solve 0.08320576664227186 lstsq 2.203964122315938e-09
